## 1. Introduction

## 2. Setup

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import timm
from torch.utils.data import DataLoader, ConcatDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

UTILS     = '/kaggle/input/datasets/tsapalyu/birdclef2026-utils'
UPSAMPLE  = '/kaggle/input/datasets/tsapalyu/birdclef2026-upsamples'
SS_CHUNKS = '/kaggle/input/datasets/tsapalyu/birdclef2026-ss-unlabeled-chunks/train_soundscapes_chunks'

SEG_DIR = f'{UPSAMPLE}/train_upsampled_species'   # audio for dataset 1

# CSV paths — adjust if uploaded to a different Kaggle dataset
SEG_CSV = f'{UPSAMPLE}/species_segment_index_PL_split.csv'
PL_CSV  = f'{SS_CHUNKS}/PL_v2s_baseline_fold2_ZERO_OUT_0.2_THRESHOLD_0.6.csv'

sys.path.append(UTILS)

from birdclef_utils.dataset import UpsampledSpeciesDataset, PseudoLabeledDataset
from birdclef_utils.constants import NUM_CLASSES

SEED = 42

CFG = {
    'model_name':    'tf_efficientnetv2_s.in21k_ft_in1k',
    'model_key':     'effnetv2s_pl',
    'batch_size':    64,
    'accum_steps':   1,
    'num_workers':   4,
    'lr':            1e-3,
    'weight_decay':  1e-5,
    'max_epochs':    20,
    'patience':      5,
    'num_classes':   NUM_CLASSES,
    'use_amp':       False,
    'bce_weight':    1.0,
    'focal_weight':  1.0,
    'focal_alpha':   0.25,
    'focal_gamma':   2.0,
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}")
print(f"Model:  {CFG['model_name']}")
print(f"Seed:   {SEED}")

## 3. Data Loading

In [ ]:
# --- label mapping ---
with open(f'{UTILS}/label2idx.json') as f:
    label2idx = json.load(f)

# --- Dataset 1: upsampled species segments (pre-split) ---
seg_df = pd.read_csv(SEG_CSV)
train_seg_df = seg_df[seg_df['split'] == 'train'].reset_index(drop=True)
val_seg_df   = seg_df[seg_df['split'] == 'val'].reset_index(drop=True)

print(f"Dataset 1 (upsampled species)")
print(f"  train: {len(train_seg_df):,}  |  val: {len(val_seg_df):,}  |  {seg_df['assigned_species'].nunique()} species")

# --- Dataset 2: pseudo-labeled soundscape chunks (90/10 split) ---
pl_df = pd.read_csv(PL_CSV)
train_pl_df, val_pl_df = train_test_split(
    pl_df, test_size=0.1, random_state=SEED, shuffle=True
)
train_pl_df = train_pl_df.reset_index(drop=True)
val_pl_df   = val_pl_df.reset_index(drop=True)

print(f"\nDataset 2 (pseudo-labeled chunks)")
print(f"  train: {len(train_pl_df):,}  |  val: {len(val_pl_df):,}  |  {pl_df.shape[1]-1} species cols")

## 4. DataLoaders

In [ ]:
def build_loaders(batch_size=64, num_workers=4):
    # Dataset 1: upsampled species (UpsampledSpeciesDataset)
    train_ds1 = UpsampledSpeciesDataset(train_seg_df, SEG_DIR, label2idx, mode='train')
    val_ds1   = UpsampledSpeciesDataset(val_seg_df,   SEG_DIR, label2idx, mode='val')

    # Dataset 2: pseudo-labeled soundscape chunks (PseudoLabeledDataset)
    train_ds2 = PseudoLabeledDataset(train_pl_df, SS_CHUNKS, label2idx, mode='train')
    val_ds2   = PseudoLabeledDataset(val_pl_df,   SS_CHUNKS, label2idx, mode='val')

    # Combine both datasets
    train_ds = ConcatDataset([train_ds1, train_ds2])
    val_ds   = ConcatDataset([val_ds1,   val_ds2])

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    return train_loader, val_loader


# Sanity check
train_loader, val_loader = build_loaders(
    batch_size=CFG['batch_size'], num_workers=CFG['num_workers']
)
specs, labels = next(iter(train_loader))
print(f"spec batch:  {specs.shape}")
print(f"label batch: {labels.shape}")
print(f"avg species per sample: {labels.sum(dim=1).mean():.2f}")
print(f"train batches: {len(train_loader)}  |  val batches: {len(val_loader)}")

## 5. Model Definition

In [ ]:
def build_model(cfg):
    model = timm.create_model(
        cfg['model_name'],
        pretrained=True,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    return model.to(DEVICE)

## 6. Training

### Loss Function

In [ ]:
class CombinedBCEFocalLoss(nn.Module):
    """1.0 · BCEWithLogitsLoss + 1.0 · torchvision SigmoidFocalLoss."""
    def __init__(self, bce_weight=1.0, focal_weight=1.0, alpha=0.25, gamma=2.0):
        super().__init__()
        self.bce_weight   = bce_weight
        self.focal_weight = focal_weight
        self.alpha        = alpha
        self.gamma        = gamma
        self.bce          = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss   = self.bce(logits, targets)
        focal_loss = torchvision.ops.sigmoid_focal_loss(
            inputs=logits, targets=targets,
            alpha=self.alpha, gamma=self.gamma, reduction='mean',
        )
        return self.bce_weight * bce_loss + self.focal_weight * focal_loss


def build_criterion(cfg):
    return CombinedBCEFocalLoss(
        bce_weight=cfg.get('bce_weight', 1.0),
        focal_weight=cfg.get('focal_weight', 1.0),
        alpha=cfg.get('focal_alpha', 0.25),
        gamma=cfg.get('focal_gamma', 2.0),
    )

### Metrics

In [ ]:
def macro_roc_auc(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        aucs.append(roc_auc_score(col, y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')


def macro_f1(y_true, y_pred, threshold=0.5):
    y_true   = np.asarray(y_true)
    y_binary = (np.asarray(y_pred) >= threshold).astype(int)
    valid = y_true.sum(axis=0) > 0
    if not valid.any():
        return float('nan')
    return float(f1_score(y_true[:, valid], y_binary[:, valid],
                          average='macro', zero_division=0))

### Train / Evaluate Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, cfg):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (specs, labels) in enumerate(loader):
        specs  = specs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model(specs)
        loss   = criterion(logits, labels)

        (loss / cfg['accum_steps']).backward()
        if (step + 1) % cfg['accum_steps'] == 0:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item()

    return running_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for specs, labels in loader:
        specs = specs.to(DEVICE, non_blocking=True)
        probs = torch.sigmoid(model(specs))
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.vstack(all_preds), np.vstack(all_labels)

### Main Training Loop

In [ ]:
def train(cfg):
    model_key = cfg['model_key']
    print(f"{'='*55}")
    print(f"{model_key.upper()}")
    print(f"{'='*55}")

    train_loader, val_loader = build_loaders(
        batch_size=cfg['batch_size'], num_workers=cfg['num_workers']
    )

    print(f"Train samples: {len(train_loader.dataset):,}")
    print(f"Val   samples: {len(val_loader.dataset):,}")

    model     = build_model(cfg)
    criterion = build_criterion(cfg)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['max_epochs'],
    )

    ckpt_path  = f'/kaggle/working/{model_key}_last.pth'
    start_epoch        = 0
    best_auc           = 0.0
    epochs_no_improve  = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch       = ckpt['epoch'] + 1
        best_auc          = ckpt['best_auc']
        epochs_no_improve = ckpt['epochs_no_improve']
        print(f"Resuming from epoch {start_epoch} (best AUC so far {best_auc:.4f})")
    else:
        print(f"No checkpoint found — training from scratch.")

    for epoch in range(start_epoch, cfg['max_epochs']):
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, cfg)

        preds, labels = evaluate(model, val_loader)
        val_auc = macro_roc_auc(labels, preds)
        val_f1  = macro_f1(labels, preds)

        scheduler.step()

        print(f"  epoch {epoch:2d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | val F1 {val_f1:.4f} | {time.time()-t0:.0f}s")

        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_auc':             best_auc,
            'epochs_no_improve':    epochs_no_improve,
            'val_auc':              val_auc,
            'model_name':           cfg['model_name'],
            'model_key':            model_key,
            'num_classes':          cfg['num_classes'],
            'in_chans':             1,
        }, ckpt_path)

        if val_auc > best_auc:
            best_auc = val_auc
            epochs_no_improve = 0
            torch.save(
                {'model_state_dict': model.state_dict(),
                 'epoch': epoch, 'val_auc': val_auc},
                f'/kaggle/working/{model_key}_best.pth',
            )
            print(f"    -> new best ({best_auc:.4f})")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg['patience']:
            print(f"  early stopping (best val AUC {best_auc:.4f})")
            break

    return best_auc

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    smoke_cfg = dict(CFG, max_epochs=1)
    print("SMOKE TEST: one epoch.")
    train(smoke_cfg)
else:
    best = train(CFG)
    print(f"\nBest val AUC: {best:.4f}")

## 7. Evaluation

In [ ]:
def load_best_model(cfg):
    model_key = cfg['model_key']
    ckpt_path = f'/kaggle/working/{model_key}_best.pth'
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = timm.create_model(
        cfg['model_name'],
        pretrained=False,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    print(f"Loaded best checkpoint from epoch {ckpt['epoch']}, val AUC {ckpt['val_auc']:.4f}")
    return model


model = load_best_model(CFG)
_, val_loader = build_loaders(batch_size=CFG['batch_size'], num_workers=CFG['num_workers'])
preds, labels = evaluate(model, val_loader)

final_auc = macro_roc_auc(labels, preds)
final_f1  = macro_f1(labels, preds)
print(f"Final val macro ROC-AUC : {final_auc:.4f}")
print(f"Final val macro F1 (0.5): {final_f1:.4f}")

pd.DataFrame([{'model': CFG['model_key'], 'val_AUC': final_auc, 'val_F1': final_f1}]
).to_csv('/kaggle/working/eval_summary.csv', index=False)
print("Saved eval_summary.csv")